# Colon Cancer Precision Oncology

Ce notebook presente une analyse simple pour classifier l'etat d'un patient a partir de 6 genes d'expression genetique.

Objectif : entrainer une Logistic Regression capable de predire `normal` ou `cancer`.

## 1. Import des bibliotheques

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2. Chargement du dataset

In [ ]:
DATA_PATH = Path('../data/colon_cancer.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/colon_cancer.csv')

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Exploration rapide

In [ ]:
print('Dimensions:', df.shape)
print('Colonnes:', list(df.columns))
print('\nRepartition des classes:')
print(df['label'].value_counts())

In [ ]:
df.describe()

## 4. Verification des 6 genes

In [ ]:
features = ['M63391', 'T62947', 'D14812', 'T51250', 'H66976', 'X55362']
target = 'label'

missing = [column for column in features + [target] if column not in df.columns]
if missing:
    raise ValueError(f'Colonnes manquantes: {missing}')

print('Toutes les colonnes necessaires sont presentes.')

## 5. Visualisation simple des classes

In [ ]:
class_counts = df['label'].value_counts()

plt.figure(figsize=(5, 4))
class_counts.plot(kind='bar', color=['#0f766e', '#be123c'])
plt.title('Repartition des classes')
plt.xlabel('Classe')
plt.ylabel('Nombre de patients')
plt.xticks(rotation=0)
plt.show()

## 6. Separation train/test et encodage des labels

In [ ]:
X = df[features]
y = df[target]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

print('Train:', X_train.shape)
print('Test:', X_test.shape)
print('Classes:', list(label_encoder.classes_))

## 7. Standardisation avec StandardScaler

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 8. Entrainement Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

## 9. Evaluation du modele

In [ ]:
y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
cm_df

In [ ]:
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## 10. Lecture des coefficients

In [ ]:
coef_df = pd.DataFrame({
    'gene': features,
    'coefficient': model.coef_[0],
}).sort_values('coefficient', key=abs, ascending=False)

coef_df

## 11. Pourquoi ces 6 genes ?

Dans ce projet, les 6 genes sont traites comme un panel reduit de biomarqueurs. L'interet pedagogique est de montrer un pipeline complet avec peu de variables : chargement des donnees, pretraitement, entrainement, evaluation et deploiement.

Un nombre limite de genes rend le modele plus simple a expliquer pendant une presentation. En pratique clinique, le choix des genes devrait etre valide par des analyses biologiques, statistiques et medicales plus completes.